In [1]:
from src.paths import RAW_DIR, PREPROCESSED_DIR, assert_data_root
from src.io_utils import save_parquet
from src.id_casting import normalize_ids
from src.schemas import TABLE_ADD_ACADEMIC_INFO
import pandas as pd

# Data-root guard (governance contract 12).
assert_data_root(RAW_DIR / "v_add_academic_info.parquet")

df = pd.read_parquet(RAW_DIR / "v_add_academic_info.parquet")

In [2]:
df.head()

,student_id,diploma_gpa,diploma_type_id,diploma_state_id,diploma_country_sl,active
0,1.111,60.00,13.111,13.0,سورية,شهادة ثانوية تجارية
1,2.111,56.82,16.111,15.0,سورية,شهادة ثانوية أدبي
2,3.111,60.45,16.111,4.0,سورية,شهادة ثانوية أدبي
3,4.111,50.83,15.111,4.0,سورية,شهادة ثانوية علمي
4,5.111,67.27,13.111,5.0,سورية,شهادة ثانوية تجارية


In [3]:
df['diploma_type_id'].value_counts()

diploma_type_id
15.111    29078
16.111     1557
51.111      952
13.111      529
19.111      129
52.111       58
26.111       51
14.111       31
25.111       24
10.111       17
20.111       16
60.111       11
65.111        9
9.111         7
31.111        6
32.111        6
69.111        6
21.111        5
18.111        5
59.111        5
70.111        5
58.111        4
63.111        3
61.111        3
27.111        2
22.111        2
71.111        2
68.111        1
Name: count, dtype: int64

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32548 entries, 0 to 32547
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          32548 non-null  float64
 1   diploma_gpa         32080 non-null  float64
 2   diploma_type_id     32524 non-null  float64
 3   diploma_state_id    25811 non-null  float64
 4   diploma_country_sl  32524 non-null  str    
 5   active              32524 non-null  str    
dtypes: float64(4), str(2)
memory usage: 3.0 MB


In [5]:
df[df['diploma_type_id'].isna()]

,student_id,diploma_gpa,diploma_type_id,diploma_state_id,diploma_country_sl,active
349,350.111,NaN,NaN,NaN,NaN,NaN
350,351.111,NaN,NaN,NaN,NaN,NaN
540,542.111,NaN,NaN,NaN,NaN,NaN
1535,1539.111,NaN,NaN,NaN,NaN,NaN
3201,3205.111,NaN,NaN,NaN,NaN,NaN
3206,3211.111,NaN,NaN,NaN,NaN,NaN
3218,3223.111,NaN,NaN,NaN,NaN,NaN
3221,3266.111,NaN,NaN,NaN,NaN,NaN
3222,3267.111,NaN,NaN,NaN,NaN,NaN
3903,3948.111,NaN,NaN,NaN,NaN,NaN


In [6]:
df1=df.dropna(subset=['diploma_type_id'], inplace=False)

In [7]:
df1[df1['diploma_type_id'].isna()]

,student_id,diploma_gpa,diploma_type_id,diploma_state_id,diploma_country_sl,active


In [8]:
_diploma_gpa_median = df1['diploma_gpa'].median()
print(f"diploma_gpa median used for fillna: {_diploma_gpa_median}")
print(f"diploma_gpa nulls before fill     : {df1['diploma_gpa'].isna().sum()}")
df1['diploma_gpa'] = df1['diploma_gpa'].fillna(_diploma_gpa_median)
print(f"diploma_gpa nulls after fill      : {df1['diploma_gpa'].isna().sum()}")

diploma_gpa median used for fillna: 85.04
diploma_gpa nulls before fill     : 446
diploma_gpa nulls after fill      : 0


In [9]:
df1['diploma_type_id']=df1['diploma_type_id'].fillna(6.111)

In [10]:
df1.info()

<class 'pandas.DataFrame'>
Index: 32524 entries, 0 to 32547
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          32524 non-null  float64
 1   diploma_gpa         32524 non-null  float64
 2   diploma_type_id     32524 non-null  float64
 3   diploma_state_id    25809 non-null  float64
 4   diploma_country_sl  32518 non-null  str    
 5   active              32524 non-null  str    
dtypes: float64(4), str(2)
memory usage: 3.2 MB


In [11]:
df1.drop(columns=['diploma_state_id','diploma_country_sl','active'], inplace=True)

In [12]:
print(f"student_id dtype قبل التطبيع : {df1['student_id'].dtype}")
print(f"student_id عينة قبل         : {df1['student_id'].head(3).tolist()}")

_rows_before_norm = len(df1)
_unique_before    = df1['student_id'].nunique()

# In-place ID casting per src/schemas.py: student_id -> string,
# diploma_type_id -> Int64 (suffix-checked and stripped inside normalize_ids).
df1 = normalize_ids(df1, table=TABLE_ADD_ACADEMIC_INFO)

_unique_after = df1['student_id'].nunique()

print(f"student_id dtype بعد التطبيع : {df1['student_id'].dtype}")
print(f"student_id عينة بعد          : {df1['student_id'].head(3).tolist()}")
print(f"diploma_type_id dtype بعد التطبيع : {df1['diploma_type_id'].dtype}")
print(f"diploma_type_id عينة بعد          : {df1['diploma_type_id'].head(3).tolist()}")

# تأكيد 1: التطبيع ما غيّر عدد الصفوف
assert len(df1) == _rows_before_norm, (
    f"عدد الصفوف تغيّر بعد التطبيع: {_rows_before_norm} -> {len(df1)}"
)
# تأكيد 2: التطبيع ما دمج طالبين بمفتاح واحد (لا تصادم مفاتيح)
assert _unique_after == _unique_before, (
    f"عدد student_id الفريدة تغيّر بعد التطبيع: {_unique_before} -> {_unique_after}. "
    "هذا يعني إنه التطبيع دمج IDs مختلفة بمفتاح واحد — أوقف وحقّق."
)
# تأكيد 3: النوع صار string فعليًا
assert df1['student_id'].dtype == 'string', (
    f"student_id المفروض string بعد التطبيع، بس هو {df1['student_id'].dtype}"
)
# تأكيد 4: diploma_type_id صار Int64 (كود تصنيفي بلا لاحقة)
assert df1['diploma_type_id'].dtype == 'Int64', (
    f"diploma_type_id المفروض Int64 بعد التطبيع، بس هو {df1['diploma_type_id'].dtype}"
)
print("تطبيع student_id و diploma_type_id: نجح (بلا فقدان صفوف، بلا تصادم مفاتيح).")

student_id dtype قبل التطبيع : float64
student_id عينة قبل         : [1.111, 2.111, 3.111]
V_ADD_ACADEMIC_INFO.diploma_type_id: uniform suffix '.111' stripped before Int64 cast
student_id dtype بعد التطبيع : string
student_id عينة بعد          : ['1.111', '2.111', '3.111']
diploma_type_id dtype بعد التطبيع : Int64
diploma_type_id عينة بعد          : [13, 16, 16]
تطبيع student_id و diploma_type_id: نجح (بلا فقدان صفوف، بلا تصادم مفاتيح).


In [13]:
save_parquet(df1, PREPROCESSED_DIR / "V_ADD_ACADEMIC_INFO" / "clean_v_add_academic_info.parquet")

WindowsPath('D:/AI/Real projects/Academic_Advisor/data/preprocessed/V_ADD_ACADEMIC_INFO/clean_v_add_academic_info.parquet')

In [14]:
df1.info()

<class 'pandas.DataFrame'>
Index: 32524 entries, 0 to 32547
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       32524 non-null  string 
 1   diploma_gpa      32524 non-null  float64
 2   diploma_type_id  32524 non-null  Int64  
dtypes: Int64(1), float64(1), string(1)
memory usage: 1.3 MB


In [15]:
df2 = pd.read_parquet(PREPROCESSED_DIR / "V_ADD_ACADEMIC_INFO" / "clean_v_add_academic_info.parquet")

In [16]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 32524 entries, 0 to 32523
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       32524 non-null  string 
 1   diploma_gpa      32524 non-null  float64
 2   diploma_type_id  32524 non-null  Int64  
dtypes: Int64(1), float64(1), string(1)
memory usage: 1.0 MB
